In [1]:
import numpy as np
import pandas as pd
import os
import wandb
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
import torch
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
import gc

In [2]:
os.environ["WANDB_API_KEY"] = "4ac0249f5d4c7d9239c6a0ec893195a256a55f46"
wandb.login()

wandb: Currently logged in as: nk23041720 (nk23041720-kathmandu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
wandb_config = {
    "model_name": "l3cube-pune/hindi-bert-v2",
    "task": "nepali_multiclass_classification",
    "learning_rate": 2e-5,  
    "batch_size": 8,
    "gradient_accumulation_steps": 2,  
    "num_epochs": 10,  
    "max_length": 256,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8, 
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "max_grad_norm": 1.0,  
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "random_seed": 42,
    "warmup_ratio": 0.1,  
    "eval_steps": 1000,  
    "logging_steps": 100,
    "save_steps": 1000,
    "early_stopping_patience": 15,
    "label_smoothing": 0.1, 
    "fp16": False,  
     "bf16": False,
}

In [4]:
run = wandb.init(
    project="topic-modeling-25k-experiments",
    name="HindiBERT-25k-2e5-16-256-10",
    config=wandb_config,
    tags=["nepali", "bert", "classification", "hindibert"]
)

In [5]:
df = pd.read_excel("/home/rupak/Desktop/Topic-Modeling /dataset/topic-modeling-25k-dataset.xlsx")
df.head()

,sentenceid,relevant_sentences,domainid
0,COM_D3E_S1_S2_6369,अक्षय कोषको आकार एक करोड रूपिया पुर्‍याउने तत्...,D3E
1,COM_D3E_S1_S2_4941,जेहेन्दार विद्यार्थीलाई छात्रवृत्ति प्रदान गरि...,D3E
2,COM_D3E_S1_S2_1399,अक्षराङ्कन पद्धतिमा शुरूमा आठ ग्रेडको व्यवस्था...,D3E
3,COM_D3E_S1_S2_1390,अक्षराङ्कन लागू हुनु पहिला विद्यार्थी दुई कित्...,D3E
4,COM_D3E_S1_S2_1049,अक्सिटोसिनले अध्यात्मिक सोच बढाउनमा सहायक पाइए...,D3E


In [6]:
df = df.sample(frac=1, random_state=wandb.config.random_seed).reset_index(drop=True)
df.head()

,sentenceid,relevant_sentences,domainid
0,COM_D4C_S1_S2_4940,सन्तानेश्वरमा पनि हेलिकप्टरबाट आउन चाहने तीर्थ...,D4C
1,COM_D4C_S1_S2_5909,विभागका अनुसार शरद ऋतुका लागि उत्तम मानिने मना...,D4C
2,COM_D2H_S1_S2_2968,चीनमा अहिले सङ्क्रमण हुन सक्ने जनावर मार्न र ख...,D2H
3,D1A_S0_B1_231,यो सिजनको मकै खेती तराईमा गरिन्छ।,D1A
4,D1A_S0_B1_4467,निजी क्षेत्र र सहकारी क्षेत्रले सरकारको उक्त न...,D1A


In [7]:
unique_labels = sorted(df["domainid"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
df["labels"] = df["domainid"].map(label2id)

In [8]:
dataset = Dataset.from_pandas(df[['relevant_sentences', 'labels']])
dataset = dataset.rename_column('relevant_sentences', 'text')

In [9]:
dataset.shape

(25006, 2)

In [10]:
train_test_split = dataset.train_test_split(
    test_size=0.2,
    seed=wandb.config.random_seed,
    shuffle=True
)
train_dataset = train_test_split['train']

In [11]:
val_test_split = train_test_split['test'].train_test_split(
    test_size=0.5,
    seed=wandb.config.random_seed,
    shuffle=True
)
val_dataset = val_test_split['train']  # 10% of total
test_dataset = val_test_split['test']   # 10% of total

In [12]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [13]:
model_name = wandb.config.model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [14]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,  # Dynamic padding with data collator
        truncation=False,
        max_length=wandb.config.max_length
    )

In [15]:
dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])

Map:   0%|          | 0/20004 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

Map:   0%|          | 0/2501 [00:00<?, ? examples/s]

In [16]:
num_labels = len(unique_labels)
num_labels

5

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.torch.float32,
    device_map="auto",
    trust_remote_code=False,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hindi-bert-v2 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

In [19]:
total_steps = (len(dataset["train"]) // (wandb.config.batch_size * wandb.config.gradient_accumulation_steps)) * wandb.config.num_epochs
warmup_steps = int(total_steps * wandb.config.warmup_ratio)

In [20]:
training_args = TrainingArguments(
    output_dir="/home/rupak/Desktop/Topic-Modeling /topic modeling 25k/checkpoint/hindibert-25k-2e5-256-16-10",
    eval_strategy="steps",
    eval_steps=wandb.config.eval_steps,
    save_strategy="steps",
    save_steps=wandb.config.save_steps,
    learning_rate=wandb.config.learning_rate,
    per_device_train_batch_size=wandb.config.batch_size,
    per_device_eval_batch_size=wandb.config.batch_size,
    gradient_accumulation_steps=wandb.config.gradient_accumulation_steps,
    num_train_epochs=wandb.config.num_epochs,
    weight_decay=wandb.config.weight_decay,
    adam_beta1=wandb.config.adam_beta1,
    adam_beta2=wandb.config.adam_beta2,
    adam_epsilon=wandb.config.adam_epsilon,
    max_grad_norm=wandb.config.max_grad_norm,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_weighted",
    greater_is_better=True,
    save_total_limit=3,
    label_smoothing_factor=wandb.config.label_smoothing,
    # Enhanced logging
    logging_dir='./logs',
    logging_steps=wandb.config.logging_steps,
    logging_first_step=True,
    # Wandb integration
    report_to="wandb",
    run_name=f"nepberta-optimized-{wandb.run.id}",
    # Full precision training (FP32)
    fp16=wandb.config.fp16,
    bf16=wandb.config.bf16,
    dataloader_drop_last=False,
    eval_accumulation_steps=None,
    # Additional optimization
    dataloader_num_workers=2,
    remove_unused_columns=False,
    push_to_hub=False,
)

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Get probabilities for AUROC
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    # Calculate comprehensive metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )

    # Macro averages for unbiased evaluation
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    # Calculate AUROC (for multiclass)
    if num_labels > 2:
        labels_binarized = label_binarize(labels, classes=range(num_labels))
        try:
            auroc_weighted = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='weighted')
            auroc_macro = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='macro')
        except ValueError:
            auroc_weighted = 0.0
            auroc_macro = 0.0
    else:
        auroc_weighted = roc_auc_score(labels, probabilities[:, 1])
        auroc_macro = auroc_weighted

    metrics = {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "auroc_weighted": auroc_weighted,
        "auroc_macro": auroc_macro
    }

    return metrics

In [22]:
class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_accuracy_history = []
        self._last_train_metrics = {}



    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss and optionally calculate training accuracy
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = outputs.get("loss")

        # Calculate training accuracy periodically
        if (self.state.global_step % (self.args.logging_steps * 2) == 0 and
            labels is not None and self.state.global_step > 0):
            with torch.no_grad():
                predictions = torch.argmax(outputs.logits, dim=-1)
                accuracy = (predictions == labels).float().mean().item()
                self._last_train_metrics = {"train_accuracy": accuracy}

        return (loss, outputs) if return_outputs else loss

In [23]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=wandb.config.early_stopping_patience)]
)

/tmp/ipykernel_722386/3450341322.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [24]:
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro,Auroc Weighted,Auroc Macro
1000,1.032900,0.947597,0.853259,0.855025,0.853259,0.851334,0.855488,0.855652,0.852882,0.962143,0.962714
2000,0.421400,0.445432,0.878049,0.880904,0.878049,0.878342,0.881808,0.879221,0.879365,0.973573,0.973817
3000,0.301100,0.450417,0.878848,0.878897,0.878848,0.877972,0.879582,0.880512,0.879183,0.977029,0.977357
4000,0.161100,0.488657,0.884046,0.883753,0.884046,0.883447,0.884493,0.885509,0.884566,0.975362,0.975774
5000,0.271100,0.583199,0.868453,0.871425,0.868453,0.865724,0.871354,0.871267,0.867275,0.973492,0.974007
6000,0.135400,0.557719,0.884846,0.885373,0.884846,0.883924,0.885382,0.886983,0.885016,0.976790,0.977174
7000,0.134000,0.608824,0.879648,0.880531,0.879648,0.879794,0.881595,0.880993,0.880995,0.973313,0.973699
8000,0.087700,0.708999,0.874050,0.875300,0.874050,0.872350,0.875126,0.876572,0.873583,0.973993,0.974347
9000,0.074100,0.717269,0.878049,0.877926,0.878049,0.876970,0.878245,0.880091,0.878186,0.971020,0.971574
10000,0.053200,0.741201,0.880848,0.880876,0.880848,0.879573,0.881357,0.882832,0.880854,0.970811,0.971289


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

TrainOutput(global_step=12510, training_loss=0.2637085182643641, metrics={'train_runtime': 2969.0173, 'train_samples_per_second': 67.376, 'train_steps_per_second': 4.214, 'total_flos': 3366600744047856.0, 'train_loss': 0.2637085182643641, 'epoch': 10.0})

In [25]:
test_results = trainer.evaluate(eval_dataset=dataset["test"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name:<20}: {value:.4f}")

Loss                : 0.6818
Accuracy            : 0.8968
Precision Weighted  : 0.8965
Recall Weighted     : 0.8968
F1 Weighted         : 0.8963
Precision Macro     : 0.8950
Recall Macro        : 0.8948
F1 Macro            : 0.8945
Auroc Weighted      : 0.9754
Auroc Macro         : 0.9750
Runtime             : 2.9908
Samples Per Second  : 836.2340
Steps Per Second    : 104.6550


In [27]:
# Get detailed predictions for test set
test_predictions = trainer.predict(dataset["test"])
test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids
test_pred_labels = np.argmax(test_logits, axis=-1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
# Convert back to original domain labels for interpretability
test_pred_domains = [id2label[pred] for pred in test_pred_labels]
test_true_domains = [id2label[label] for label in test_labels]

In [29]:
report = classification_report(
    test_labels, 
    test_pred_labels, 
    target_names=[id2label[i] for i in range(num_labels)],
    digits=4
)

In [30]:
print(report)

              precision    recall  f1-score   support

         D1A     0.9332    0.9174    0.9253       533
         D2H     0.8560    0.8930    0.8741       486
         D3E     0.9074    0.9108    0.9091       527
         D4C     0.9262    0.9617    0.9436       496
         D5G     0.8521    0.7908    0.8203       459

    accuracy                         0.8968      2501
   macro avg     0.8950    0.8948    0.8945      2501
weighted avg     0.8965    0.8968    0.8963      2501



In [ ]:
wandb.finish()

eval/accuracy,▁▅▅▆▃▆▅▄▅▅▆▆█
eval/auroc_macro,▁▆█▇▆█▆▇▅▅▄▅▇
eval/auroc_weighted,▁▆█▇▆█▆▇▅▅▄▅▇
eval/f1_macro,▁▅▅▆▃▆▆▄▅▆▆▇█
eval/f1_weighted,▁▅▅▆▃▆▅▄▅▅▆▆█
eval/loss,█▁▁▂▃▃▃▅▅▅▅▅▄
eval/precision_macro,▁▆▅▆▄▆▆▄▅▆▆▆█
eval/precision_weighted,▁▅▅▆▄▆▅▄▅▅▆▆█
eval/recall_macro,▁▅▅▆▄▇▆▅▅▆▇▇█
eval/recall_weighted,▁▅▅▆▃▆▅▄▅▅▆▆█
+21,...


: 